In [ ]:
import glob
import os
import mne
import numpy as np

def get_file_pairs(data_dir):
    # Find all PSG files
    psg_files = sorted(glob.glob(os.path.join(data_dir, "*PSG.edf")))
    # Find all Hypnogram files
    hypno_files = sorted(glob.glob(os.path.join(data_dir, "*Hypnogram.edf")))
    
    pairs = []
    for psg in psg_files:
        # Extract the ID (e.g., SC4001) to find the matching hypnogram
        file_id = os.path.basename(psg)[:6]
        matching_hypno = [h for h in hypno_files if file_id in h]
        
        if matching_hypno:
            pairs.append((psg, matching_hypno[0]))
            
    return pairs

# Example usage:
pairs = get_file_pairs("../../dataset/sleepEDF/physionet.org/files/sleep-edfx/1.0.0/sleep-cassette/")
pairs

[('../../data/sleepEDF/physionet.org/files/sleep-edfx/1.0.0/sleep-cassette/SC4001E0-PSG.edf',
  '../../data/sleepEDF/physionet.org/files/sleep-edfx/1.0.0/sleep-cassette/SC4001EC-Hypnogram.edf'),
 ('../../data/sleepEDF/physionet.org/files/sleep-edfx/1.0.0/sleep-cassette/SC4002E0-PSG.edf',
  '../../data/sleepEDF/physionet.org/files/sleep-edfx/1.0.0/sleep-cassette/SC4002EC-Hypnogram.edf'),
 ('../../data/sleepEDF/physionet.org/files/sleep-edfx/1.0.0/sleep-cassette/SC4011E0-PSG.edf',
  '../../data/sleepEDF/physionet.org/files/sleep-edfx/1.0.0/sleep-cassette/SC4011EH-Hypnogram.edf'),
 ('../../data/sleepEDF/physionet.org/files/sleep-edfx/1.0.0/sleep-cassette/SC4012E0-PSG.edf',
  '../../data/sleepEDF/physionet.org/files/sleep-edfx/1.0.0/sleep-cassette/SC4012EC-Hypnogram.edf'),
 ('../../data/sleepEDF/physionet.org/files/sleep-edfx/1.0.0/sleep-cassette/SC4021E0-PSG.edf',
  '../../data/sleepEDF/physionet.org/files/sleep-edfx/1.0.0/sleep-cassette/SC4021EH-Hypnogram.edf'),
 ('../../data/sleepEDF/ph

In [43]:
test_pair = pairs[40]
psg_file, hyp_file = test_pair

raw = mne.io.read_raw_edf(psg_file, preload=True, verbose=False)
annots = mne.read_annotations(hyp_file)
raw.set_annotations(annots, emit_warning=False)

/tmp/ipykernel_141752/1850511453.py:4: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(psg_file, preload=True, verbose=False)
/tmp/ipykernel_141752/1850511453.py:4: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(psg_file, preload=True, verbose=False)
/tmp/ipykernel_141752/1850511453.py:4: RuntimeWarning: Highpass cutoff frequency 16.0 is greater than lowpass cutoff frequency 0.7, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_edf(psg_file, preload=True, verbose=False)


<RawEDF | SC4202E0-PSG.edf, 7 x 8010000 (80100.0 s), ~427.8 MiB, data loaded>

In [44]:
raw.pick_channels(["EEG Fpz-Cz"])
# raw.pick_channels(channels)
raw.filter(l_freq=0.5, h_freq=30.0, )

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 661 samples (6.610 s)



<RawEDF | SC4202E0-PSG.edf, 1 x 8010000 (80100.0 s), ~61.1 MiB, data loaded>

In [46]:
event_id = {
    'Sleep stage W': 0,
    'Sleep stage 1': 1,
    'Sleep stage 2': 2,
    'Sleep stage 3': 3,
    'Sleep stage 4': 3, # Merge Stage 3 and 4 into "Deep Sleep"
    'Sleep stage R': 4
}
events, _ = mne.events_from_annotations(raw, event_id=event_id, chunk_duration=30.0, verbose=False)
epochs = mne.Epochs(raw=raw, events=events, event_id=event_id, 
                    tmin=0, tmax=30.0 - 1/raw.info['sfreq'], 
                    baseline=None, preload=True, verbose=False, on_missing="ignore")

In [47]:
e = {}
for i in epochs.events[:, 2]:
    if i not in e:
        e[i] = 1
    else:
        e[i] += 1
    
print(e)

{np.int64(0): 1864, np.int64(1): 120, np.int64(2): 479, np.int64(4): 207}


In [38]:
# Find the corrupted data
for ind, i in enumerate(pairs):
    if "SC4201E0-PSG.edf" in i[0]:
        print(ind)

39


In [ ]:
def process_all_subjects(pairs, channels=["EEG Fpz-Cz"], win_size=30.0):
    all_x = []
    all_y = []
    
    # Mapping for Sleep-EDF stages to 5-class system
    event_id = {
        'Sleep stage W': 0,
        'Sleep stage 1': 1,
        'Sleep stage 2': 2,
        'Sleep stage 3': 3,
        'Sleep stage 4': 3, # Merge Stage 3 and 4 into "Deep Sleep"
        'Sleep stage R': 4
    }

    for psg_path, hypno_path in pairs:
        print(f"Processing: {os.path.basename(psg_path)}")
        
        # 1. Load Data
        raw = mne.io.read_raw_edf(psg_path, preload=True, verbose=False)
        annots = mne.read_annotations(hypno_path)
        raw.set_annotations(annots, emit_warning=False)
        
        # 2. Filter & Channel Selection
        raw.pick_channels(channels)
        raw.filter(l_freq=0.5, h_freq=win_size, verbose=False)
        
        # 3. Epoching (30s windows)
        events, _ = mne.events_from_annotations(raw, event_id=event_id, chunk_duration=win_size, verbose=False)
        epochs = mne.Epochs(raw=raw, events=events, event_id=event_id, 
                            tmin=0, tmax=win_size - 1/raw.info['sfreq'], 
                            baseline=None, preload=True, verbose=False, on_missing="ignore")
        
        # 4. Normalization & Storage
        x = epochs.get_data() # (N, 1, 3000)
        y = epochs.events[:, 2] # (N,)
        
        # Z-score normalization per subject
        x = (x - np.mean(x)) / (np.std(x) + 1e-8)
        
        all_x.append(x)
        all_y.append(y)
    
    return np.concatenate(all_x), np.concatenate(all_y)

# Final Execution
x_final, y_final = process_all_subjects(pairs)

In [58]:
x_final.shape

(414961, 1, 3000)

In [59]:
# Save multiple arrays into one compressed file
np.savez_compressed('../../dataset/sleepEDF/sleep_edf_processed.npz', x=x_final, y=y_final)

# # To load:
# data = np.load('sleep_edf_processed.npz')
# x_loaded = data['x']
# y_loaded = data['y']